# 🏥 Assistant Médical Qwen - Interface Gradio
Ce notebook charge le modèle Qwen (fine-tuné avec QLoRA) et le système RAG (FAISS) depuis Google Drive, puis lance une interface utilisateur Gradio moderne et interactive.

In [3]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install faiss-cpu sentence-transformers gradio


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0n7__j3x/unsloth_ce0a50d3e139423f9e98ad69310dc294
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0n7__j3x/unsloth_ce0a50d3e139423f9e98ad69310dc294
  Resolved https://github.com/unslothai/unsloth.git to commit aba57d0872d75e167d375db004b4d64f676bd11f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.9 MB/s eta 0:00:00

In [4]:
import os
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

# ⚠️ METTRE À JOUR CES CHEMINS SI NÉCESSAIRE ⚠️
DRIVE_BASE = "/content/drive/MyDrive/medical_qwen_project"
FINAL_MODEL_PATH = f"{DRIVE_BASE}/final_model"
FAISS_INDEX_PATH = f"{DRIVE_BASE}/rag_assets/medical_rag.faiss"
CHUNKS_PATH = f"{DRIVE_BASE}/rag_assets/rag_documents.jsonl"

print("✅ Paths configured.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Paths configured.


In [5]:
from unsloth import FastLanguageModel
import torch

print("Loading model and tokenizer...")
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = FINAL_MODEL_PATH,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
print("✅ Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model and tokenizer...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Model loaded successfully!


In [6]:
import faiss
import json
from sentence_transformers import SentenceTransformer

print("Loading RAG index and embedding model...")
# 1. Load FAISS index
if os.path.exists(FAISS_INDEX_PATH):
    index = faiss.read_index(FAISS_INDEX_PATH)
    print(f"✅ FAISS Index loaded (Total vectors: {index.ntotal})")
else:
    print(f"❌ ERROR: FAISS Index not found at {FAISS_INDEX_PATH}")

# 2. Load Text Chunks from JSONL
medical_chunks = []
if os.path.exists(CHUNKS_PATH):
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                # On extrait le texte selon la structure de ton jsonl (généralement 'text', 'content' ou 'page_content')
                chunk_text = data.get('text', data.get('page_content', data.get('content', str(data))))
                medical_chunks.append(chunk_text)
    print(f"✅ Medical chunks loaded (Total chunks: {len(medical_chunks)})")
else:
    print(f"❌ ERROR: Chunks file not found at {CHUNKS_PATH}")

# 3. Load Embedding Model (must match the one used during indexing)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded.")


Loading RAG index and embedding model...
✅ FAISS Index loaded (Total vectors: 10100)
✅ Medical chunks loaded (Total chunks: 10100)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded.


In [7]:
import torch

MEDICAL_WHITELIST = [
    "sick", "ill", "illness", "pain", "ache", "aching",
    "hurt", "hurting", "sore", "tired", "fatigue", "fatigued",
    "weak", "weakness", "fever", "high fever", "low fever", "chills",
    "cold sweat", "sweating", "night sweats", "cough", "dry cough", "wet cough",
    "persistent cough", "sneeze", "sneezing", "nausea", "vomit", "vomiting",
    "dizzy", "dizziness", "vertigo", "faint", "fainting", "swollen",
    "swelling", "bleed", "bleeding", "rash", "itch", "itching",
    "burn", "burning", "numb", "numbness", "tingling", "pins and needles",
    "shortness of breath", "breathing", "breath", "wheezing", "hyperventilation", "swallow",
    "difficulty swallowing", "chest pain", "stomach pain", "abdominal pain", "headache", "migraine",
    "cluster headache", "tension headache", "cramp", "muscle cramp", "twitch", "tremor",
    "shaking", "seizure", "convulsion", "blurry vision", "double vision", "vision loss",
    "hearing loss", "ringing ears", "taste loss", "smell loss", "runny nose", "stuffy nose",
    "congestion", "sinus pressure", "heartburn", "indigestion", "bloating", "gas",
    "constipation", "diarrhea", "loose stool", "bloody stool", "black stool", "loss of appetite",
    "increased appetite", "dehydration", "dry mouth", "mouth ulcer", "canker sore", "sensitivity",
    "palpitations", "rapid heartbeat", "slow heartbeat", "irregular heartbeat", "frozen shoulder", "stiff neck",
    "back pain", "lower back pain", "upper back pain", "joint pain", "muscle pain", "bone pain",
    "hip pain", "knee pain", "ankle pain", "foot pain", "hand pain", "wrist pain",
    "elbow pain", "toothache", "jaw pain", "ear pain", "eye pain", "pelvic pain",
    "groin pain", "fatty stool", "urinary urgency", "frequent urination", "painful urination", "blood in urine",
    "pus", "infection", "pus discharge", "discharge", "vaginal discharge", "penile discharge",
    "itchy skin", "dry skin", "peeling skin", "bruising", "bruises", "hair loss",
    "balding", "weight loss", "weight gain", "memory loss", "confusion", "hallucination",
    "panic attack", "mood swings", "irritability", "aggression", "brain fog", "difficulty concentrating",
    "attention problems", "speech difficulty", "slurred speech", "paralysis", "head", "scalp",
    "forehead", "temple", "face", "jaw", "chin", "neck",
    "throat", "tongue", "mouth", "lip", "gums", "tooth",
    "teeth", "ear", "eardrum", "eye", "eyelid", "retina",
    "pupil", "nose", "nostril", "sinus", "shoulder", "arm",
    "forearm", "wrist", "hand", "finger", "thumb", "nail",
    "chest", "rib", "lung", "heart", "artery", "vein",
    "blood vessel", "abdomen", "stomach", "intestine", "bowel", "colon",
    "rectum", "anus", "liver", "gallbladder", "pancreas", "spleen",
    "kidney", "bladder", "ureter", "urethra", "pelvis", "hip",
    "groin", "back", "spine", "vertebra", "disc", "muscle",
    "tendon", "ligament", "joint", "cartilage", "bone", "marrow",
    "knee", "leg", "thigh", "calf", "shin", "ankle",
    "heel", "foot", "toe", "brain", "nerve", "spinal cord",
    "skin", "hair", "immune system", "lymph node", "thyroid", "appendix",
    "prostate", "uterus", "ovary", "testicle", "cervix", "vagina",
    "penis", "breast", "nipple", "disease", "disorder", "condition",
    "syndrome", "viral infection", "bacterial infection", "fungal infection", "parasite", "sepsis",
    "cancer", "tumor", "benign tumor", "malignant tumor", "diabetes", "type 1 diabetes",
    "type 2 diabetes", "prediabetes", "asthma", "allergy", "allergic reaction", "eczema",
    "psoriasis", "acne", "hypertension", "high blood pressure", "low blood pressure", "cholesterol",
    "high cholesterol", "thyroid disorder", "hypothyroidism", "hyperthyroidism", "anemia", "arthritis",
    "osteoarthritis", "rheumatoid arthritis", "osteoporosis", "alzheimer", "dementia", "parkinson",
    "epilepsy", "covid", "covid19", "flu", "influenza", "cold",
    "pneumonia", "bronchitis", "tuberculosis", "stroke", "heart attack", "obesity",
    "metabolic syndrome", "anxiety", "panic disorder", "depression", "bipolar", "schizophrenia",
    "ptsd", "ocd", "adhd", "autism", "insomnia", "sleep apnea",
    "narcolepsy", "ulcer", "gastritis", "acid reflux", "gerd", "crohn disease",
    "ulcerative colitis", "ibs", "kidney stone", "liver disease", "hepatitis", "cirrhosis",
    "fatty liver", "migraine disorder", "multiple sclerosis", "lupus", "fibromyalgia", "lyme disease",
    "malaria", "dengue", "zika", "hiv", "aids", "std",
    "sti", "gonorrhea", "syphilis", "chlamydia", "herpes", "hpv",
    "genital warts", "endometriosis", "pcos", "infertility", "erectile dysfunction", "cataract",
    "glaucoma", "macular degeneration", "deafness", "autoinflammatory disease", "autoimmune disease", "genetic disorder",
    "doctor", "physician", "surgeon", "specialist", "cardiologist", "neurologist",
    "dermatologist", "psychiatrist", "psychologist", "therapist", "dentist", "orthodontist",
    "pediatrician", "gynecologist", "urologist", "oncologist", "radiologist", "nurse",
    "paramedic", "pharmacist", "hospital", "clinic", "emergency room", "urgent care",
    "ambulance", "intensive care", "icu", "surgery", "operation", "procedure",
    "biopsy", "transplant", "xray", "x-ray", "mri", "ct scan",
    "ultrasound", "ecg", "ekg", "blood test", "urine test", "diagnosis",
    "diagnose", "screening", "checkup", "physical exam", "lab result", "scan",
    "monitoring", "vaccination", "stitches", "cast", "dialysis", "chemotherapy",
    "radiation therapy", "immunotherapy", "rehabilitation", "physical therapy", "occupational therapy", "treatment",
    "therapy", "medication", "medicine", "drug", "pill", "tablet",
    "capsule", "syrup", "ointment", "cream", "injection", "dose",
    "dosage", "prescription", "vitamin", "supplement", "vaccine", "antibiotic",
    "painkiller", "antidepressant", "antihistamine", "insulin", "ibuprofen", "paracetamol",
    "acetaminophen", "aspirin", "naproxen", "amoxicillin", "penicillin", "metformin",
    "omeprazole", "prednisone", "antiviral", "antifungal", "sedative", "anesthesia",
    "pregnant", "pregnancy", "trimester", "baby", "infant", "newborn",
    "birth", "labor", "delivery", "miscarriage", "stillbirth", "fertility",
    "ovulation", "menstrual", "menstruation", "period", "heavy period", "cramps",
    "menopause", "breastfeed", "breastfeeding", "contraception", "birth control", "c-section",
    "weight", "diet", "nutrition", "exercise", "fitness", "sleep",
    "hydration", "calorie", "protein", "healthy", "unhealthy", "chronic",
    "acute", "emergency", "wellness", "mental health", "public health", "lifestyle",
    "recovery", "rehab", "self care", "stress management", "wheelchair", "crutches",
    "walker", "brace", "bandage", "gauze", "mask", "oxygen tank",
    "ventilator", "pacemaker", "hearing aid", "glucose monitor", "thermometer", "stethoscope",
    "blood pressure monitor", "defibrillator", "symptom", "diagnostic", "pathology", "prognosis",
    "epidemic", "pandemic", "contagious", "infectious", "immune response", "antibody",
    "inflammation", "chronic pain", "acute pain", "relapse", "remission", "side effect",
    "adverse reaction", "medical history", "family history", "risk factor", "preventive care", "cyanosis",
    "hypoxia", "hypoxemia", "tachycardia", "bradycardia", "arrhythmia", "tachypnea",
    "dyspnea", "apnea", "hemorrhage", "hematoma", "edema", "erythema",
    "lesion", "ulceration", "necrosis", "ischemia", "embolism", "thrombosis",
    "aneurysm", "vasculitis", "neuropathy", "myopathy", "atrophy", "spasticity",
    "ataxia", "paresis", "hemiplegia", "quadriplegia", "paraplegia", "syncope",
    "delirium", "aphasia", "dysphagia", "dysarthria", "photophobia", "phonophobia",
    "hematuria", "proteinuria", "ketosis", "acidosis", "alkalosis", "jaundice",
    "cyanotic", "pallor", "cachexia", "cachectic", "anorexia", "bulimia",
    "malnutrition", "deconditioning", "hyperglycemia", "hypoglycemia", "hyperlipidemia", "hypercalcemia",
    "hypocalcemia", "hypernatremia", "hyponatremia", "hyperkalemia", "hypokalemia", "decompensation",
    "exacerbation", "idiopathic", "iatrogenic", "comorbidity", "contraindication", "prophylaxis",
    "analgesia", "sedation", "intubation", "extubation", "resuscitation", "triage",
    "catheter", "cannula", "suction", "aspiration", "incision", "excision",
    "laparoscopy", "endoscopy", "colonoscopy", "bronchoscopy", "angiography", "echocardiogram",
    "mammogram", "pap smear", "spirometry", "audiometry", "electrolyte", "hemoglobin",
    "platelet", "leukocyte", "erythrocyte", "hematocrit", "creatinine", "bilirubin",
    "triglyceride", "glucose", "insomnia disorder", "somnolence", "fatigability", "psychosis",
    "mania", "hypomania", "catatonia", "anhedonia", "derealization", "depersonalization",
    "suicidality", "self-harm", "addiction", "dependency", "withdrawal", "tolerance",
    "overdose", "intoxication", "poisoning", "toxicity", "allergen", "histamine",
    "anaphylaxis", "urticaria", "rhinitis", "sinusitis", "otitis", "tonsillitis",
    "laryngitis", "pharyngitis", "conjunctivitis", "cellulitis", "dermatitis", "folliculitis",
    "impetigo", "scabies", "ringworm", "candidiasis", "meningitis", "encephalitis",
    "myocarditis", "pericarditis", "endocarditis", "atherosclerosis", "cardiomyopathy", "angina",
    "ischemic", "hemorrhagic stroke", "transient ischemic attack", "copd", "emphysema", "sarcoidosis",
    "fibrosis", "pulmonary edema", "pleurisy", "pneumothorax", "atelectasis", "hepatomegaly",
    "splenomegaly", "pancreatitis", "cholecystitis", "appendicitis", "diverticulitis", "peritonitis",
    "nephritis", "pyelonephritis", "cystitis", "glomerulonephritis", "hydronephrosis", "urolithiasis",
    "osteomyelitis", "scoliosis", "kyphosis", "lordosis", "bursitis", "tendinitis",
    "sciatica", "carpal tunnel", "gout", "spondylosis", "fracture", "sprain",
    "strain", "dislocation", "laceration", "abrasion", "contusion", "concussion",
    "trauma", "burn injury", "frostbite", "heatstroke", "heat exhaustion", "dehydrated",
    "hypothermia", "hyperthermia", "septic shock", "cardiogenic shock", "hypovolemic shock", "multiorgan failure",
    "respiratory failure", "cardiac arrest", "respiratory arrest", "coma", "vegetative state", "palliative care",
    "hospice", "mortality", "morbidity", "prenatal", "postpartum", "neonatal",
    "gestational diabetes", "preeclampsia", "eclampsia", "placenta previa", "ectopic pregnancy", "amenorrhea",
    "dysmenorrhea", "mastitis", "gynecomastia", "vasectomy", "hysterectomy", "prostatectomy",
    "circumcision", "cryotherapy", "electrotherapy", "hydrotherapy", "telemedicine", "telehealth",
    "epidural", "lumbar puncture", "bone marrow biopsy", "blood transfusion", "plasma", "serum",
    "pathogen", "microbe", "bacterium", "virus", "fungus", "protozoa",
    "helminth", "microbiome", "genome", "mutation", "chromosome", "hereditary",
    "congenital", "prenatal screening", "immunodeficiency", "autoantibody", "cytokine", "antigen",
    "vaccinated", "immunized", "booster shot", "quarantine", "isolation", "contact tracing",
    "outbreak", "endemic", "epidemiology", "biomarker", "clinical trial", "placebo",
    "double blind", "randomized trial", "evidence-based", "pharmacology", "toxicology", "oncology",
    "hematology", "endocrinology", "gastroenterology", "rheumatology", "nephrology", "pulmonology",
    "geriatrics", "pediatrics", "obstetrics", "anesthesiology", "pathophysiology", "histology",
    "cytology", "radiology", "sonography", "electrocardiography", "electroencephalogram", "emg",
    "pet scan", "dexa scan", "fluoroscopy", "biostatistics", "breathe", "blurry",
    "vision", "hearing", "taste", "smell", "blood", "allergic",
    "stress", "test", "taken", "take", "taking", "took",
    "prescribed", "prescribe", "mg", "ml", "medicament", "medicin",
    "feel", "feeling", "felt", "suffer", "suffering", "inject",
    "inhaler", "inhale", "morphine", "codeine", "tramadol", "oxycodone",
    "fentanyl", "diclofenac", "azithromycin", "doxycycline", "lisinopril", "atorvastatin",
    "metoprolol", "amlodipine", "sertraline", "fluoxetine", "diazepam", "cetirizine",
    "loratadine", "cortisone", "salbutamol", "warfarin", "heparin", "methotrexate",
    "hydroxychloroquine", "tinnitus", "cataracts", "i feel", "i am feeling", "i have been",
    "i've been", "i've had", "i have a", "i had a", "my body", "my chest",
    "my stomach", "my head", "hurts", "burning sensation", "sharp pain", "dull pain",
    "constant pain", "comes and goes", "getting worse", "since yesterday", "for days", "for weeks",
    "for months", "not feeling well", "feel unwell", "feel awful", "feel terrible", "can't sleep",
    "can't breathe", "can't eat", "can't walk", "lost weight", "gained weight", "losing hair",
    "losing appetite", "oxycontin", "percocet", "vicodin", "norco", "xanax",
    "valium", "ativan", "klonopin", "ambien", "zoloft", "prozac",
    "lexapro", "effexor", "wellbutrin", "adderall", "ritalin", "concerta",
    "strattera", "lipitor", "crestor", "zocor", "pravastatin", "losartan",
    "valsartan", "atenolol", "propranolol", "bisoprolol", "pantoprazole", "lansoprazole",
    "esomeprazole", "levothyroxine", "synthroid", "methimazole", "albuterol", "budesonide",
    "fluticasone", "salmeterol", "gabapentin", "pregabalin", "topiramate", "lamotrigine",
    "clonidine", "guanfacine", "methylphenidate", "hydroxyzine", "diphenhydramine", "promethazine",
    "ondansetron", "metoclopramide", "domperidone", "ranitidine", "famotidine", "cimetidine",
    "dexamethasone", "betamethasone", "hydrocortisone", "colchicine", "allopurinol", "febuxostat",
    "sildenafil", "tadalafil", "vardenafil", "tamsulosin", "finasteride", "dutasteride",
    "clopidogrel", "aspirin therapy", "rivaroxaban", "apixaban", "acyclovir", "valacyclovir",
    "oseltamivir", "remdesivir", "doxorubicin", "cyclophosphamide", "carboplatin", "taxol",
    "child", "children", "kid", "toddler", "pediatric", "growth chart",
    "developmental delay", "milestone", "immunization", "diaper rash", "colic", "cradle cap",
    "jaundice newborn", "febrile seizure", "ear infection", "strep throat", "rsv", "chickenpox",
    "measles", "mumps", "rubella", "whooping cough", "elderly", "geriatric",
    "senior", "aging", "age-related", "fall risk", "hip fracture", "nursing home",
    "memory care", "dementia care", "incontinence", "bedsore", "pressure ulcer", "mental",
    "psychiatric", "psychological", "behavioral", "therapy session", "counseling", "psychotherapy",
    "cbt", "eating disorder", "binge eating", "purging", "body image", "suicidal thoughts",
    "self harm", "cutting", "substance abuse", "alcohol abuse", "drug abuse", "rehabilitation center",
    "detox", "withdrawal symptoms", "relapse prevention", "calorie deficit", "calorie surplus", "bmi",
    "body mass index", "metabolism", "blood sugar", "glucose level", "a1c", "glycemia",
    "ketogenic", "intermittent fasting", "deficiency", "iron deficiency", "vitamin d deficiency", "calcium deficiency",
    "omega 3", "probiotics", "gut health", "sexually transmitted", "sexual health", "libido",
    "impotence", "premature ejaculation", "vaginismus", "dyspareunia", "ovarian cyst", "uterine fibroid",
    "polycystic", "endometrium", "sperm count", "egg freezing", "ivf", "surrogacy",
    "mole", "wart", "keloid", "scar", "stretch mark", "hyperpigmentation",
    "vitiligo", "rosacea", "seborrhea", "blackhead", "whitehead", "pore",
    "collagen", "botox", "myopia", "hyperopia", "astigmatism", "presbyopia",
    "contact lens", "laser surgery", "lasik", "retinal detachment", "strabismus", "amblyopia",
    "dry eyes", "floaters", "glasses", "spectacles", "lens", "lenses",
    "contacts", "glaces", "optometrist", "ophthalmologist", "optician", "eye test",
    "cavity", "root canal", "crown", "filling", "extraction", "gum disease",
    "gingivitis", "periodontitis", "plaque", "braces", "wisdom tooth", "jaw clicking",
    "tmj", "urgent", "911", "ambulance needed", "unconscious", "not breathing",
    "choking", "drowning", "poisoned", "overdosed", "severe bleeding", "can't move",
]

REFUSAL_MSG = "I'm a specialized medical assistant and can only answer health-related questions. Please ask me about symptoms, medications, or health concerns."

GREETINGS = {
    "hello", "hi", "hey", "salut", "bonjour",
    "good morning", "good evening", "howdy", "sup", "hola"
}

def is_medical(text):
    """Returns True only if at least one medical keyword is present."""
    lower = text.lower()
    return any(kw in lower for kw in MEDICAL_WHITELIST)

SYSTEM_PROMPT = """You are a specialized medical AI assistant.
Answer only health-related questions using the retrieved context when relevant.
Be empathetic, clear, and never claim certainty. Recommend consulting a doctor for serious concerns.
"""

def retrieve_context(query, top_k=3):
    """Fetches relevant context from the FAISS database."""
    query_vector = embedding_model.encode([query])
    distances, indices = index.search(query_vector, top_k)

    context_parts = []
    for i, idx in enumerate(indices[0]):
        if idx != -1 and idx < len(medical_chunks):
            context_parts.append(f"Document {i+1}:\n{medical_chunks[idx]}")

    if context_parts:
        return "\n\n---\n\n".join(context_parts)
    return ""

def medical_chat(user_message, history):
    """Main function to handle chat requests."""

    # Free GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    user_msg_clean = user_message.strip().lower()

    # 1. Greeting intercept
    if user_msg_clean in GREETINGS:
        history.append([user_message, "Hello! I am your specialized medical AI assistant. How can I help you with your health today?"])
        yield history
        return

    # 2. Whitelist check — block EVERYTHING that is not clearly medical
    if not is_medical(user_message):
        history.append([user_message, REFUSAL_MSG])
        yield history
        return

    # 3. Medical query — retrieve context and call the model
    retrieved_context = retrieve_context(user_message)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for turn in history:
        messages.append({"role": "user", "content": turn[0]})
        messages.append({"role": "assistant", "content": turn[1]})

    if retrieved_context:
        augmented_message = f"Retrieved Medical Context:\n{retrieved_context}\n\nUser Query: {user_message}"
    else:
        augmented_message = user_message

    messages.append({"role": "user", "content": augmented_message})

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    from transformers import TextIteratorStreamer
    from threading import Thread

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        input_ids=input_ids,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.3,
        top_p=0.9,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    history.append([user_message, ""])
    generated_text = ""
    for new_text in streamer:
        generated_text += new_text
        history[-1][1] = generated_text
        yield history


In [ ]:
# ============================================================
#  DROP-IN REPLACEMENT FOR THE GRADIO CELL
#  Paste this as the LAST cell in your Colab notebook.
# ============================================================

# ── 1. Install FastAPI + Uvicorn ──────────────
!pip install fastapi uvicorn nest-asyncio pyngrok -q


# ── 3. FastAPI app ───────────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from transformers import TextIteratorStreamer
from threading import Thread
import asyncio, json, uvicorn

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

# ── Serve the HTML frontend ──────────────────────────────────
@app.get("/", response_class=HTMLResponse)
async def serve_ui():
    with open("/content/drive/MyDrive/medical_qwen_project/ui.html", encoding="utf-8") as f:
        return HTMLResponse(f.read())

# ── Streaming /chat endpoint ─────────────────────────────────
@app.post("/chat")
async def chat_endpoint(request: Request):
    body      = await request.json()
    user_msg  = body.get("message", "")
    history   = body.get("history", [])

    # ── RAG retrieval ──────────────────────────────────────────
    retrieved_context = retrieve_context(user_msg)

    # ── Build message list ─────────────────────────────────────
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history:
        messages.append({"role": "user",      "content": turn[0]})
        messages.append({"role": "assistant", "content": turn[1]})

    if retrieved_context:
        augmented = (
            f"Context information is below.\n\n"
            f"---------------------\n{retrieved_context}\n---------------------\n\n"
            f"Given the context information and not prior knowledge, answer the following query:\n\n{user_msg}"
        )
    else:
        augmented = user_msg

    messages.append({"role": "user", "content": augmented})

    # ── Tokenise ───────────────────────────────────────────────
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    # ── Streamer ───────────────────────────────────────────────
    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True,
    )
    generation_kwargs = dict(
        input_ids=input_ids,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.3,
        top_p=0.9,
    )
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # ── SSE generator ──────────────────────────────────────────
    async def token_stream():
        loop = asyncio.get_event_loop()
        for token in streamer:
            safe = token.replace("\n", "\\n")
            yield f"data: {safe}\n\n"
            await asyncio.sleep(0)
        yield "data: [DONE]\n\n"

    return StreamingResponse(token_stream(), media_type="text/event-stream")

# ── 4. Expose via ngrok & launch ─────────────────────────────
from pyngrok import ngrok

ngrok.set_auth_token("3Dp0gpGF9q9u70nbctfTm6RhD5E_2bAiLjbAuM3ttoW2frrZF")

public_url = ngrok.connect(8000)
print("=" * 55)
print(f"  🏥  Medical Assistant UI is live at:")
print(f"  👉  {public_url}")
print("=" * 55)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config=config)
asyncio.run(server.serve())


  🏥  Medical Assistant UI is live at:
  👉  NgrokTunnel: "https://wrongful-malformed-confider.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [2561]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     196.64.242.187:0 - "GET / HTTP/1.1" 200 OK
INFO:     196.64.242.187:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     196.64.242.187:0 - "POST /chat HTTP/1.1" 200 OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


INFO:     196.64.242.187:0 - "POST /chat HTTP/1.1" 200 OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     196.64.242.187:0 - "GET / HTTP/1.1" 200 OK
INFO:     196.64.242.187:0 - "POST /chat HTTP/1.1" 200 OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
